In [ ]:
# Install runtime dependencies.
%pip install -U \
  "torch" \
  "torchvision" \
  "transformers==5.12.1" \
  "accelerate" \
  "safetensors" \
  "peft" \
  "pillow==12.3.0" \
  "fastapi" \
  "uvicorn[standard]" \
  "requests" \
  "python-multipart"

In [ ]:
import sys
sys.path.insert(0, "../")

from pathlib import Path

# Notebook is expected to be inside the project repo.
PROJECT_DIR = Path("../").resolve()
DATA_DIR = PROJECT_DIR.parent / "data"
CHECKPOINT_ARCHIVE_DIR = DATA_DIR / "gigachat_vl_archive"
CHECKPOINT_DIR = DATA_DIR / "gigachat_vl_checkpoint"

CHECKPOINT_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("CHECKPOINT_ARCHIVE_DIR:", CHECKPOINT_ARCHIVE_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

In [ ]:
import requests
from tqdm.auto import tqdm

# Download checkpoint parts from a public Yandex Disk folder.
PUBLIC_URL = "https://disk.yandex.ru/d/07nH6MDVArT5DA"

PART_NAMES = [
    "gigachat_checkpoint_7500.zip.part.aa",
    "gigachat_checkpoint_7500.zip.part.ab",
    "gigachat_checkpoint_7500.zip.part.ac",
]

def get_yadisk_download_url(public_url: str, path: str) -> str:
    response = requests.get(
        "https://cloud-api.yandex.net/v1/disk/public/resources/download",
        params={"public_key": public_url, "path": path},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["href"]

def download_file(url: str, dst_path: Path, chunk_size: int = 1024 * 1024) -> None:
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))

        with open(dst_path, "wb") as f, tqdm(
            total=total if total > 0 else None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=dst_path.name,
        ) as pbar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if not chunk:
                    continue
                f.write(chunk)
                pbar.update(len(chunk))

for part_name in PART_NAMES:
    dst_path = CHECKPOINT_ARCHIVE_DIR / part_name
    if dst_path.exists() and dst_path.stat().st_size > 0:
        print(f"Already exists: {dst_path}")
        continue

    url = get_yadisk_download_url(PUBLIC_URL, f"/{part_name}")
    download_file(url, dst_path)

In [ ]:
# Merge parts and unpack the checkpoint.
import zipfile

ARCHIVE_PATH = DATA_DIR / "gigachat_checkpoint_7500.zip"

with open(ARCHIVE_PATH, "wb") as archive:
    for name in PART_NAMES:
        part_path = PARTS_DIR / name
        with open(part_path, "rb") as part:
            while True:
                chunk = part.read(1024 * 1024 * 32)
                if not chunk:
                    break
                archive.write(chunk)

print("archive:", ARCHIVE_PATH, ARCHIVE_PATH.stat().st_size)

with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    zf.extractall(CHECKPOINTS_DIR)

print("unpacked to:", CHECKPOINTS_DIR)

In [ ]:
# Find the checkpoint directory.
def find_checkpoint_dir(root: Path) -> Path:
    candidates = sorted(root.rglob("vlm_meta.json"))
    candidates = [p.parent for p in candidates if (p.parent / "projector.pt").exists()]
    if not candidates:
        raise FileNotFoundError(f"No checkpoint with vlm_meta.json and projector.pt under {root}")
    return candidates[0]

CKPT_DIR = find_checkpoint_dir(CHECKPOINTS_DIR)
print("checkpoint:", CKPT_DIR)

In [ ]:
model = GigaChatVLForInference(
    checkpoint_dir=str(CKPT_DIR),
    use_4bit_llm=False,
    device="cuda:0",
)
model.eval()

print("model loaded")

In [ ]:
# Start the project API server.
import threading
import time

import uvicorn
import src.api.server as server_module

server_module.model = model

config = uvicorn.Config(
    server_module.app,
    host="0.0.0.0",
    port=8000,
    log_level="info",
)

uvicorn_server = uvicorn.Server(config)
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()

time.sleep(3)
print("server: http://127.0.0.1:8000")

In [ ]:
# Stop the server when needed.
uvicorn_server.should_exit = True

In [ ]:
from pathlib import Path

MERA_DIR = DATA_DIR / "MERA_MULTIMODAL"
WORK_DIR = DATA_DIR / "gigachat_vl_eval"

RESULTS_DIR = WORK_DIR / "results"
DATASETS_CACHE_DIR = WORK_DIR / "ds_cache"
REQ_CACHE_DIR = WORK_DIR / "request_cache"

for p in [DATA_DIR, RESULTS_DIR, DATASETS_CACHE_DIR, REQ_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR =", PROJECT_DIR)
print("DATA_DIR =", DATA_DIR)
print("MERA_DIR =", MERA_DIR)
print("WORK_DIR =", WORK_DIR)

In [ ]:
import subprocess

# Clone MERA multimodal benchmark repo.
if not MERA_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--recurse-submodules",
            "https://github.com/MERA-Evaluation/MERA_MULTIMODAL.git",
            str(MERA_DIR),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "pull"], cwd=MERA_DIR, check=True)
    subprocess.run(["git", "submodule", "update", "--init", "--recursive"], cwd=MERA_DIR, check=True)

subprocess.run(["git", "submodule", "update", "--init", "--recursive"], cwd=MERA_DIR, check=True)

In [ ]:
# Install MERA dependencies.
%pip install -U pip setuptools wheel
%pip install -e "$MERA_DIR/lm-evaluation-harness[api]" pillow numpy sentencepiece

In [ ]:
from pathlib import Path

# Patch optional media/model imports.
def patch_api_model():
    path = MERA_DIR / "lm-evaluation-harness/lm_eval/api/model.py"
    text = path.read_text(encoding="utf-8")

    if "AudioDecoder = None" not in text:
        text = text.replace(
            "from datasets.features._torchcodec import AudioDecoder\n",
            "try:\n    from datasets.features._torchcodec import AudioDecoder\nexcept Exception:\n    AudioDecoder = None\n",
        )

    if "VideoDecoder = None" not in text:
        text = text.replace(
            "from torchcodec.decoders._video_decoder import VideoDecoder\n",
            "try:\n    from torchcodec.decoders._video_decoder import VideoDecoder\nexcept Exception:\n    VideoDecoder = None\n",
        )

    if "decord = None" not in text:
        text = text.replace(
            "import decord\n",
            "try:\n    import decord\nexcept Exception:\n    decord = None\n",
        )

    if "tvio = None" not in text:
        text = text.replace(
            "import torchvision.io as tvio\n",
            "try:\n    import torchvision.io as tvio\nexcept Exception:\n    tvio = None\n",
        )

    text = text.replace(
        "if isinstance(obj, AudioDecoder):",
        "if AudioDecoder is not None and isinstance(obj, AudioDecoder):",
    )
    text = text.replace(
        "if isinstance(obj, VideoDecoder):",
        "if VideoDecoder is not None and isinstance(obj, VideoDecoder):",
    )
    text = text.replace(
        "if isinstance(obj, decord.VideoReader):",
        "if decord is not None and isinstance(obj, decord.VideoReader):",
    )
    text = text.replace(
        "if isinstance(obj, tvio.VideoReader):",
        "if tvio is not None and isinstance(obj, tvio.VideoReader):",
    )

    path.write_text(text, encoding="utf-8")


def patch_models_utils():
    path = MERA_DIR / "lm-evaluation-harness/lm_eval/models/utils.py"
    text = path.read_text(encoding="utf-8")

    if "MistralTokenizer = None" not in text:
        text = text.replace(
            "from vllm.transformers_utils.tokenizers.mistral import MistralTokenizer\n",
            "try:\n    from vllm.transformers_utils.tokenizers.mistral import MistralTokenizer\nexcept Exception:\n    MistralTokenizer = None\n",
        )

    text = text.replace(
        "if isinstance(tokenizer, MistralTokenizer):",
        "if MistralTokenizer is not None and isinstance(tokenizer, MistralTokenizer):",
    )

    path.write_text(text, encoding="utf-8")


def patch_models_init():
    path = MERA_DIR / "lm-evaluation-harness/lm_eval/models/__init__.py"
    path.write_text(
        """from importlib import import_module

_MODEL_MODULES = [
    "anthropic_llms",
    "api_models",
    "dummy",
    "gguf",
    "hf_audiolm",
    "hf_steered",
    "hf_videolm",
    "hf_vlms",
    "huggingface",
    "ibm_watsonx_ai",
    "mamba_lm",
    "nemo_lm",
    "neuron_optimum",
    "openai_completions",
    "optimum_ipex",
    "optimum_lm",
    "sglang_causallms",
    "sglang_generate_API",
    "textsynth",
    "vllm_causallms",
    "vllm_vlms",
]

for _module_name in _MODEL_MODULES:
    try:
        globals()[_module_name] = import_module(f".{_module_name}", __name__)
    except Exception:
        pass

try:
    import hf_transfer
    import huggingface_hub.constants

    huggingface_hub.constants.HF_HUB_ENABLE_HF_TRANSFER = True
except ImportError:
    pass
""",
        encoding="utf-8",
    )


def patch_load_media():
    path = MERA_DIR / "multimodal_tasks/load_media.py"
    text = path.read_text(encoding="utf-8")

    text = text.replace("import soundfile as sf\n", "")
    text = text.replace(
        "def get_audio(audio_json):\n    if load_bytes or load_base64 or load_files:\n",
        "def get_audio(audio_json):\n    if load_bytes or load_base64 or load_files:\n        import soundfile as sf\n\n",
    )

    path.write_text(text, encoding="utf-8")


patch_api_model()
patch_models_utils()
patch_models_init()
patch_load_media()

print("Patched MERA files for image-only inference")

In [ ]:
import subprocess
import sys

# Check patched imports.
subprocess.run(
    [sys.executable, "-c", "import lm_eval.api.model; import lm_eval.models; print('imports ok')"],
    cwd=MERA_DIR,
    check=True,
)

In [ ]:
import os

# MERA runtime config.
os.environ["HF_DATASETS_CACHE"] = str(DATASETS_CACHE_DIR)
os.environ["LOAD_BASE64"] = "1"
os.environ["OPENAI_API_KEY"] = "EMPTY"

MODEL_NAME = "gigachat_vl"
BASE_URL = "http://127.0.0.1:8000/v1/chat/completions"
TASK_NAME = "ruclevr"

CACHE_FILE = REQ_CACHE_DIR / f"{TASK_NAME}_{MODEL_NAME}.sqlite"
OUT_DIR = RESULTS_DIR / f"{TASK_NAME}_{MODEL_NAME}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_NAME =", MODEL_NAME)
print("BASE_URL =", BASE_URL)
print("TASK_NAME =", TASK_NAME)
print("CACHE_FILE =", CACHE_FILE)
print("OUT_DIR =", OUT_DIR)

In [ ]:
# Run MERA task.
%cd {MERA_DIR}

!lm-eval \
  --model openai-chat-completions \
  --model_args model={MODEL_NAME},base_url={BASE_URL},num_concurrent=1,max_retries=3,timeout=90000 \
  --output_path="{OUT_DIR}" \
  --batch_size=1 \
  --log_samples \
  --seed 1234 \
  --num_fewshot=0 \
  --apply_chat_template \
  --fewshot_as_multiturn \
  --pass_multimodal_args_to_chat_history \
  --use_cache "{CACHE_FILE}" \
  --include_path ./multimodal_tasks \
  --tasks {TASK_NAME}